In [1]:
from pyspark.sql import SparkSession
from datetime import datetime


In [2]:
spark = SparkSession.builder.appName("myAPP").getOrCreate()

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from pyspark.sql.functions import col, trim
from datetime import datetime

In [4]:
orders_data = [
 ("O001","Delhi ","Laptop","45000","2024-01-05","Completed"),
 ("O002","Mumbai","Mobile ","32000","05/01/2024","Completed"),
 ("O003","Bangalore","Tablet","30000","2024/01/06","Completed"),
 ("O004","Delhi","Laptop","","2024-01-07","Cancelled"),
 ("O005","Mumbai","Mobile","invalid","2024-01-08","Completed"),
 ("O006","Chennai","Tablet",None,"2024-01-08","Completed"),
 ("O007","Delhi","Laptop","47000","09-01-2024","Completed"),
 ("O008","Bangalore","Mobile","28000","2024-01-09","Completed"),
 ("O009","Mumbai","Laptop","55000","2024-01-10","Completed"),
 ("O009","Mumbai","Laptop","55000","2024-01-10","Completed")
]

In [5]:
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("city", StringType(), True),
    StructField("product", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("status", StringType(), True)
])

In [6]:
df = spark.createDataFrame(orders_data, schema)

In [7]:
df = df.withColumn("city", trim(col("city"))) \
       .withColumn("product", trim(col("product")))

In [8]:

def parse_amount(val):
    try:
        return int(val)
    except:
        return None

parse_amount_udf = spark.udf.register("parse_amount_udf", parse_amount, IntegerType())
df = df.withColumn("amount", parse_amount_udf(col("amount")))


def parse_date(val):
    formats = ["%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d", "%d-%m-%Y"]
    for fmt in formats:
        try:
            return datetime.strptime(val, fmt).date()
        except:
            continue
    return None


In [9]:
parse_date_udf = spark.udf.register("parse_date_udf", parse_date, DateType())
df = df.withColumn("order_date", parse_date_udf(col("order_date")))


In [10]:
df = df.dropDuplicates()

df_clean = df.filter(col("status") == "Completed")

df_clean.show()
df_clean.printSchema()

+--------+---------+-------+------+----------+---------+
|order_id|     city|product|amount|order_date|   status|
+--------+---------+-------+------+----------+---------+
|    O002|   Mumbai| Mobile| 32000|2024-01-05|Completed|
|    O001|    Delhi| Laptop| 45000|2024-01-05|Completed|
|    O003|Bangalore| Tablet| 30000|2024-01-06|Completed|
|    O009|   Mumbai| Laptop| 55000|2024-01-10|Completed|
|    O007|    Delhi| Laptop| 47000|2024-01-09|Completed|
|    O008|Bangalore| Mobile| 28000|2024-01-09|Completed|
|    O005|   Mumbai| Mobile|  NULL|2024-01-08|Completed|
|    O006|  Chennai| Tablet|  NULL|2024-01-08|Completed|
+--------+---------+-------+------+----------+---------+

root
 |-- order_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- status: string (nullable = true)



# Total revenue per city

In [11]:
from pyspark.sql.functions import sum, avg

revenue_city = df_clean.groupBy("city").agg(sum("amount").alias("total_revenue"))
revenue_city.show()


+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|        58000|
|  Chennai|         NULL|
|   Mumbai|        87000|
|    Delhi|        92000|
+---------+-------------+



# Total revenue per product

In [12]:


revenue_product = df_clean.groupBy("product").agg(sum("amount").alias("total_revenue"))
revenue_product.show()



+-------+-------------+
|product|total_revenue|
+-------+-------------+
| Laptop|       147000|
| Mobile|        60000|
| Tablet|        30000|
+-------+-------------+



# Average order value per city

In [13]:

avg_order_city = df_clean.groupBy("city").agg(avg("amount").alias("avg_order_value"))
avg_order_city.show()

+---------+---------------+
|     city|avg_order_value|
+---------+---------------+
|Bangalore|        29000.0|
|  Chennai|           NULL|
|   Mumbai|        43500.0|
|    Delhi|        46000.0|
+---------+---------------+



# Windor Partitions

In [14]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

windowSpec = Window.orderBy(col("total_revenue").desc())

ranked_cities = revenue_city.withColumn("rank", rank().over(windowSpec))
ranked_cities.show()



+---------+-------------+----+
|     city|total_revenue|rank|
+---------+-------------+----+
|    Delhi|        92000|   1|
|   Mumbai|        87000|   2|
|Bangalore|        58000|   3|
|  Chennai|         NULL|   4|
+---------+-------------+----+



# Identify top-performing city

In [15]:

top_city = ranked_cities.filter(col("rank") == 1)
top_city.show()

+-----+-------------+----+
| city|total_revenue|rank|
+-----+-------------+----+
|Delhi|        92000|   1|
+-----+-------------+----+



# Cache the cleaned DataFrame

In [16]:
df_clean.cache()

DataFrame[order_id: string, city: string, product: string, amount: int, order_date: date, status: string]

# Run two aggregations and observe behavior

In [17]:
df_clean.groupBy("city").agg(sum("amount").alias("total_revenue")).show()
df_clean.groupBy("product").agg(sum("amount").alias("total_revenue")).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|  Chennai|         NULL|
|   Mumbai|        87000|
|    Delhi|        92000|
|Bangalore|        58000|
+---------+-------------+

+-------+-------------+
|product|total_revenue|
+-------+-------------+
| Tablet|        30000|
| Laptop|       147000|
| Mobile|        60000|
+-------+-------------+



# Use explain(True) to inspect the plan

In [18]:
df_clean.groupBy("city").agg(sum("amount").alias("total_revenue")).explain(True)

== Parsed Logical Plan ==
'Aggregate ['city], ['city, 'sum('amount) AS total_revenue#543]
+- Filter (status#5 = Completed)
   +- Deduplicate [city#6, order_id#0, amount#9, product#7, order_date#11, status#5]
      +- Project [order_id#0, city#6, product#7, amount#9, parse_date_udf(order_date#4)#10 AS order_date#11, status#5]
         +- Project [order_id#0, city#6, product#7, parse_amount_udf(amount#3)#8 AS amount#9, order_date#4, status#5]
            +- Project [order_id#0, city#6, trim(product#2, None) AS product#7, amount#3, order_date#4, status#5]
               +- Project [order_id#0, trim(city#1, None) AS city#6, product#2, amount#3, order_date#4, status#5]
                  +- LogicalRDD [order_id#0, city#1, product#2, amount#3, order_date#4, status#5], false

== Analyzed Logical Plan ==
city: string, total_revenue: bigint
Aggregate [city#6], [city#6, sum(amount#9) AS total_revenue#543L]
+- Filter (status#5 = Completed)
   +- Deduplicate [city#6, order_id#0, amount#9, product#7